In [47]:
import pandas as pd
df = pd.read_csv("../data/BMW/clean_reviews.csv")
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time I tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,I have a vehicle with comfort access and a Sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,Can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,BMW stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,The lack of support for Octopus Intelligent Go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


##### Wir entfernen im nächsten Schritt html

In [48]:
import re 
def remove_html(text):
    return re.sub(r"<.*?>", "", text)
df["content"]=df["content"].apply(remove_html)
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time I tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,I have a vehicle with comfort access and a Sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,Can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,BMW stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,The lack of support for Octopus Intelligent Go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


##### Nachfolgend wird die Spalte "at" in "datetime"-Format umgewandelt.

In [49]:
df["at"]=pd.to_datetime(df["at"])

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11008 entries, 0 to 11007
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              11008 non-null  object        
 1   content               11008 non-null  object        
 2   score                 11008 non-null  int64         
 3   thumbsUpCount         11008 non-null  int64         
 4   reviewCreatedVersion  10429 non-null  object        
 5   at                    11008 non-null  datetime64[ns]
 6   appVersion            10429 non-null  object        
 7   company               11008 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 688.1+ KB


##### Im nächsten Schritt werden reviews mit weniger als 2 Wörtern werden entfernt, da sie für unser NLP Modell keinen besonderen Mehrwert liefern.

In [51]:
df=df[df["content"].str.split().str.len()>=2]

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10509 entries, 0 to 11007
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              10509 non-null  object        
 1   content               10509 non-null  object        
 2   score                 10509 non-null  int64         
 3   thumbsUpCount         10509 non-null  int64         
 4   reviewCreatedVersion  9953 non-null   object        
 5   at                    10509 non-null  datetime64[ns]
 6   appVersion            9953 non-null   object        
 7   company               10509 non-null  object        
dtypes: datetime64[ns](1), int64(2), object(5)
memory usage: 738.9+ KB


##### Wir setzen die Texte in lowercases

In [53]:
df["content"]=df["content"].str.lower()

In [54]:
df.head()

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW


##### Nun müssten wir die Punctuation entfernen. Dies haben wir jedoch bereit in der Explorativen Datenanalyse erledigt. Im nächsten Schritt machen wir mit der Tokenization weiter.

In [55]:
df["clean_text"]=df["content"].str.lower()
import re
df["clean_text"]=df["clean_text"].apply(lambda x: re.sub(r"[^a-zA-Z0-9äöüÄÖÜß\s]", "", x))
def tokenize(text):
    return text.split()
df["tokens"]=df["clean_text"].apply(tokenize)
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time i tap on the widget to open the app...,"[every, time, i, tap, on, the, widget, to, ope..."
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,i have a vehicle with comfort access and a sam...,"[i, have, a, vehicle, with, comfort, access, a..."
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add my e34 and e39,"[cant, add, my, e34, and, e39]"
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands for its reliability n performance i...,"[bmw, stands, for, its, reliability, n, perfor..."
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,the lack of support for octopus intelligent go...,"[the, lack, of, support, for, octopus, intelli..."
...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,auf dem ersten blick sehe ich keine vorteile i...,"[auf, dem, ersten, blick, sehe, ich, keine, vo..."
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design aber wesentliche funktionen der ...,"[nettes, design, aber, wesentliche, funktionen..."
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app vom schreibtisch aus kann ich ...,"[klasse, neue, app, vom, schreibtisch, aus, ka..."
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick aber warum fehlen funktionen die e...,"[ganz, schick, aber, warum, fehlen, funktionen..."


##### Im nachfolgenden Schritt entfernen wir die Stopwords. In der Tabelle sehen wir unter der Spalte "tokens" den Unterschied.

In [56]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("german") + stopwords.words("english"))
df["tokens"]=df["tokens"].apply(lambda words: [word for word in words if word not in stop_words])
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time i tap on the widget to open the app...,"[every, time, tap, widget, open, app, starts, ..."
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,i have a vehicle with comfort access and a sam...,"[vehicle, comfort, access, samsung, galaxy, s2..."
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add my e34 and e39,"[cant, add, e34, e39]"
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands for its reliability n performance i...,"[bmw, stands, reliability, n, performance, roa..."
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,the lack of support for octopus intelligent go...,"[lack, support, octopus, intelligent, go, frus..."
...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,auf dem ersten blick sehe ich keine vorteile i...,"[ersten, blick, sehe, vorteile, fand, bisherig..."
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design aber wesentliche funktionen der ...,"[nettes, design, wesentliche, funktionen, alte..."
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app vom schreibtisch aus kann ich ...,"[klasse, neue, app, schreibtisch, lüftung, bmw..."
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick aber warum fehlen funktionen die e...,"[ganz, schick, warum, fehlen, funktionen, alte..."


##### Im nachfolgenden Schritt wandeln wir tokens wieder zu Text.

In [57]:
df["clean_text"]=df["tokens"].apply(lambda words: " ".join(words))
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time tap widget open app starts remote h...,"[every, time, tap, widget, open, app, starts, ..."
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung, galaxy, s2..."
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add e34 e39,"[cant, add, e34, e39]"
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands reliability n performance road ill ...,"[bmw, stands, reliability, n, performance, roa..."
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus..."
...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,ersten blick sehe vorteile fand bisherige dars...,"[ersten, blick, sehe, vorteile, fand, bisherig..."
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design wesentliche funktionen alten app...,"[nettes, design, wesentliche, funktionen, alte..."
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app schreibtisch lüftung bmw einsc...,"[klasse, neue, app, schreibtisch, lüftung, bmw..."
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick warum fehlen funktionen alten app ...,"[ganz, schick, warum, fehlen, funktionen, alte..."


##### Wir speichern das finale NLP Dataset

In [58]:
df.to_csv("../data/BMW/preprocessed_reviews.csv", index=False)

##### Wir bauen jetzt Lemmatization für englisch und deutsch. Was macht LEmmatization?
Lemmatization reduziert Wörter auf ihre Grundform.
- Beispiel
- Original: connecting --> Lemma: connect
- Original: connected --> Lemma: connect
- Original: connections --> Lemma: connection
- Original: funktioniert --> Lemma: funktionieren
- Original: funktionierte --> Lemma: funktionieren

Da unsere Reviews gemischt sind, brauchen wir Englisch Lemmatizer und Deutsch Lemmatizer. Die beste Lösung hierzu ist spaCy.
Hierzu laden wir im Terminal spaCy herunter (pip install spacy). Dann die Sprachmodelle (python -m spacy download en_core_web_sm) und (python -m spacy download de_core_news_sm)

In [59]:
import spacy
nlp_en = spacy.load("en_core_web_sm")
nlp_de=spacy.load("de_core_news_sm")

##### Anschließend bauen wir eine Funktion für beide Sprachen

In [ ]:
#def lemmatize_text(text):
    #doc_en = nlp_en(text)
    #doc_de = nlp_de(text)
    #lemmas_en = [token.lemma_ for token in doc_en]
    #lemmas_de = [token.lemma_ for token in doc_de]
    #return " ".join(lemmas_en + lemmas_de)

Was jetzt hier oben passiert? Wir lassen beide Modelle englisch und deutsch über den selben Text laufen.
- Beispiel Review: die app funktioniert nicht
- Englischer Lemmatizer: die app funktioniert nicht
- Deutscher Lemmatizer: der app funktioniert nicht
- Unser Ergebnis: die app funktioniert nicht der app funktionieren nicht
Das erzeugt also doppelte Tokens, falsche Statistik und schlechtere Modelle sowie schlechtere Topics.
Richtige Lösung ist, nur einen Lemmatizer pro text zu verwenden. Das geht über LAnguage Detection. Hierzu installieren wir erst langdetect (pip install langdetect)

In [60]:
from langdetect import detect
def lemmatizer_text(text):
    try:
        lang = detect(text)
    except:
        lang="en"
    if lang=="de":
        doc = nlp_de(text)
    else:
        doc = nlp_en(text)
    lemmas = [token.lemma_ for token in doc]
    return " ".join(lemmas)


##### Nun wenden wir diesen an

In [61]:
df["lemmatized_text"]=df["clean_text"].apply(lemmatizer_text)
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens,lemmatized_text
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time tap widget open app starts remote h...,"[every, time, tap, widget, open, app, starts, ...",every time tap widget open app start remote he...
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung, galaxy, s2...",vehicle comfort access samsung galaxy s26 ultr...
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add e34 e39,"[cant, add, e34, e39]",can not add e34 e39
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands reliability n performance road ill ...,"[bmw, stands, reliability, n, performance, roa...",bmw stand reliability n performance road ill h...
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus...",lack support octopus intelligent go frustratin...
...,...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,ersten blick sehe vorteile fand bisherige dars...,"[ersten, blick, sehe, vorteile, fand, bisherig...",erster Blick sehen Vorteil finden bisherig Dar...
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design wesentliche funktionen alten app...,"[nettes, design, wesentliche, funktionen, alte...",nettes Design wesentlich Funktion Alt app fehl...
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app schreibtisch lüftung bmw einsc...,"[klasse, neue, app, schreibtisch, lüftung, bmw...",Klasse neu app schreibtisch Lüftung bmw einsch...
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick warum fehlen funktionen alten app ...,"[ganz, schick, warum, fehlen, funktionen, alte...",ganz Schick warum fehlen Funktion alten app ge...


##### Nun speichern wir das Dataset erneut nach der Lemmatization

In [62]:
df.to_csv("../data/BMW/preprocessed_reviews.csv", index=False)

##### Das Preprocessing Notebook ist hiermit abgeschlossen.

Optional können wir Bigram Detection einsetzen. Aktuell sieht der text so aus (remote start work) Für ein Modell sind das einzelne Wörter. Aber viele Begriffe gehören eigentlich zusammen (remote_start, connected_drive, not_work, login_problem). Das ist extrem wichtig für 
- Topic Modeling
- Issue Detection
- Clustering
- Embeddings
- RAG
Das Modell versteht das Problem viel bessser.

In [63]:
from gensim.models.phrases import Phrases, Phraser
sentences = df["tokens"]
bigram= Phrases(sentences, min_count=5, threshold=10)
bigram_model = Phraser(bigram)
df["tokens_bigram"]=df["tokens"].apply(lambda x: bigram_model[x])
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens,lemmatized_text,tokens_bigram
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time tap widget open app starts remote h...,"[every, time, tap, widget, open, app, starts, ...",every time tap widget open app start remote he...,"[every_time, tap, widget, open, app, starts, r..."
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung, galaxy, s2...",vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung_galaxy, s26..."
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add e34 e39,"[cant, add, e34, e39]",can not add e34 e39,"[cant_add, e34, e39]"
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands reliability n performance road ill ...,"[bmw, stands, reliability, n, performance, roa...",bmw stand reliability n performance road ill h...,"[bmw, stands, reliability, n, performance, roa..."
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus...",lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus..."
...,...,...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,ersten blick sehe vorteile fand bisherige dars...,"[ersten, blick, sehe, vorteile, fand, bisherig...",erster Blick sehen Vorteil finden bisherig Dar...,"[ersten, blick, sehe, vorteile, fand, bisherig..."
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design wesentliche funktionen alten app...,"[nettes, design, wesentliche, funktionen, alte...",nettes Design wesentlich Funktion Alt app fehl...,"[nettes, design, wesentliche, funktionen_alten..."
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app schreibtisch lüftung bmw einsc...,"[klasse, neue, app, schreibtisch, lüftung, bmw...",Klasse neu app schreibtisch Lüftung bmw einsch...,"[klasse, neue, app, schreibtisch, lüftung, bmw..."
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick warum fehlen funktionen alten app ...,"[ganz, schick, warum, fehlen, funktionen, alte...",ganz Schick warum fehlen Funktion alten app ge...,"[ganz, schick, warum, fehlen, funktionen_alten..."


##### Token dann wieder zu text wandeln

In [64]:
df["final_text"]= df["tokens_bigram"].apply(lambda words: " ".join(words))
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens,lemmatized_text,tokens_bigram,final_text
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time tap widget open app starts remote h...,"[every, time, tap, widget, open, app, starts, ...",every time tap widget open app start remote he...,"[every_time, tap, widget, open, app, starts, r...",every_time tap widget open app starts remote h...
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung, galaxy, s2...",vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung_galaxy, s26...",vehicle comfort access samsung_galaxy s26 ultr...
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add e34 e39,"[cant, add, e34, e39]",can not add e34 e39,"[cant_add, e34, e39]",cant_add e34 e39
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands reliability n performance road ill ...,"[bmw, stands, reliability, n, performance, roa...",bmw stand reliability n performance road ill h...,"[bmw, stands, reliability, n, performance, roa...",bmw stands reliability n performance road ill ...
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus...",lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus...",lack support octopus intelligent go frustratin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,ersten blick sehe vorteile fand bisherige dars...,"[ersten, blick, sehe, vorteile, fand, bisherig...",erster Blick sehen Vorteil finden bisherig Dar...,"[ersten, blick, sehe, vorteile, fand, bisherig...",ersten blick sehe vorteile fand bisherige dars...
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design wesentliche funktionen alten app...,"[nettes, design, wesentliche, funktionen, alte...",nettes Design wesentlich Funktion Alt app fehl...,"[nettes, design, wesentliche, funktionen_alten...",nettes design wesentliche funktionen_alten app...
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app schreibtisch lüftung bmw einsc...,"[klasse, neue, app, schreibtisch, lüftung, bmw...",Klasse neu app schreibtisch Lüftung bmw einsch...,"[klasse, neue, app, schreibtisch, lüftung, bmw...",klasse neue app schreibtisch lüftung bmw einsc...
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick warum fehlen funktionen alten app ...,"[ganz, schick, warum, fehlen, funktionen, alte...",ganz Schick warum fehlen Funktion alten app ge...,"[ganz, schick, warum, fehlen, funktionen_alten...",ganz schick warum fehlen funktionen_alten app ...


##### Speichere Dokument separat und teste später

In [65]:
df.to_csv("../data/BMW/bigram_preprocessed_reviews.csv", index=False)

In [66]:
df

,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,appVersion,company,clean_text,tokens,lemmatized_text,tokens_bigram,final_text
0,fb7fb9f2-75ad-4660-8c9d-ad05bae0739d,every time i tap on the widget to open the app...,3,0,5.11.4,2026-03-07 16:35:07,5.11.4,BMW,every time tap widget open app starts remote h...,"[every, time, tap, widget, open, app, starts, ...",every time tap widget open app start remote he...,"[every_time, tap, widget, open, app, starts, r...",every_time tap widget open app starts remote h...
1,86b51f94-88da-42b0-87ac-fcbe94e86ea8,i have a vehicle with comfort access and a sam...,2,0,5.11.4,2026-03-07 14:42:54,5.11.4,BMW,vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung, galaxy, s2...",vehicle comfort access samsung galaxy s26 ultr...,"[vehicle, comfort, access, samsung_galaxy, s26...",vehicle comfort access samsung_galaxy s26 ultr...
2,20f31dc9-de57-46c7-b503-2fdc2ca757c4,can't add my e34 and e39 :/,1,0,5.11.4,2026-03-07 09:50:37,5.11.4,BMW,cant add e34 e39,"[cant, add, e34, e39]",can not add e34 e39,"[cant_add, e34, e39]",cant_add e34 e39
3,dffdfe4f-70a8-40f2-b684-ba6a5858fe55,bmw stands for its reliability n performance i...,5,0,5.11.3,2026-03-07 07:47:33,5.11.3,BMW,bmw stands reliability n performance road ill ...,"[bmw, stands, reliability, n, performance, roa...",bmw stand reliability n performance road ill h...,"[bmw, stands, reliability, n, performance, roa...",bmw stands reliability n performance road ill ...
4,4c00e344-ebb7-486d-b0ed-8ef2b0fa0cbb,the lack of support for octopus intelligent go...,2,0,5.11.4,2026-03-06 19:38:16,5.11.4,BMW,lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus...",lack support octopus intelligent go frustratin...,"[lack, support, octopus, intelligent, go, frus...",lack support octopus intelligent go frustratin...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11002,82b0e88c-e668-4700-a4d7-054c067444bd,"auf dem ersten blick sehe ich keine vorteile, ...",3,2,1.0.0,2020-07-28 06:42:14,1.0.0,BMW,ersten blick sehe vorteile fand bisherige dars...,"[ersten, blick, sehe, vorteile, fand, bisherig...",erster Blick sehen Vorteil finden bisherig Dar...,"[ersten, blick, sehe, vorteile, fand, bisherig...",ersten blick sehe vorteile fand bisherige dars...
11004,e2250c51-90ce-4562-a5ff-33bd7ab5fa19,"nettes design, aber wesentliche funktionen der...",2,4,NaN,2020-07-27 17:08:12,NaN,BMW,nettes design wesentliche funktionen alten app...,"[nettes, design, wesentliche, funktionen, alte...",nettes Design wesentlich Funktion Alt app fehl...,"[nettes, design, wesentliche, funktionen_alten...",nettes design wesentliche funktionen_alten app...
11005,146646a2-71e6-428d-b16e-0db1c7ffd48a,klasse neue app!! vom schreibtisch aus kann ic...,5,1,1.0.0,2020-07-27 16:50:01,1.0.0,BMW,klasse neue app schreibtisch lüftung bmw einsc...,"[klasse, neue, app, schreibtisch, lüftung, bmw...",Klasse neu app schreibtisch Lüftung bmw einsch...,"[klasse, neue, app, schreibtisch, lüftung, bmw...",klasse neue app schreibtisch lüftung bmw einsc...
11006,01c2c3a0-4aac-4b4f-9cf6-ec4995563e48,"ganz schick, aber warum fehlen funktionen, die...",1,25,1.0.0,2020-07-27 16:45:50,1.0.0,BMW,ganz schick warum fehlen funktionen alten app ...,"[ganz, schick, warum, fehlen, funktionen, alte...",ganz Schick warum fehlen Funktion alten app ge...,"[ganz, schick, warum, fehlen, funktionen_alten...",ganz schick warum fehlen funktionen_alten app ...


##### Preprocessing ist somit komplett fertig. Hinweis: Wenn Bigram Detection genutzt wird, sollten wir tokens --> tokens_bigram-->final_text verwenden und nicht mehr clean_text

##### Hinweis: Nach Projektinternem Austausch am 10.03.2026
Nach Diskussion mit Martin Lohr, haben wir uns dazu entschieden, keine Tokenization manuell vorzunehmen, da das SentenceTransformer-Modell (all-MiniLM-L6-v2) die tokenization intern übernimmt. Wenn wir manuell tokenisieren, würden wir die semantische Struktur zerstören und die Embeddings schlechter machen. Somit ist eine Textbereinigung mit HTML entfernen, lowercasing, Interpunktion löschen ausreichend.

In [69]:
df["appVersion"].value_counts()

appVersion
1.6.2     428
3.3.1     367
4.7.3     352
1.5.2     335
2.7.0     321
         ... 
3.11.4      2
3.11.0      1
5.11.5      1
5.11.0      1
4.3.0       1
Name: count, Length: 102, dtype: int64